# 02. Sentiment Modeling: Detecting Information Asymmetry

**Context**: In the Kenyan e-commerce landscape, numerical star ratings often mask qualitative dissatisfaction. This notebook builds an NLP pipeline to identify "Information Asymmetry"—cases where products maintain high ratings despite critical feedback hidden in local dialects (Sheng/Swahili).

## 2.1 Load Preprocessed Data

We pull the synthesized NLP dataset created in Phase 1. This dataset already includes slang-mapped tokens and joined product metadata.

In [7]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.data_preprocessing import DataLoader, TextPreprocessor
from src.sentiment_analysis import SentimentModel

# 1. Initialize tools
loader = DataLoader(data_dir="../data/raw/")
text_proc = TextPreprocessor()

# 2. Build the Base NLP Frame
_, prods, revs = loader.load_all()
nlp_df = text_proc.build_nlp_frame(revs, prods)

# 3. Load and Clean Jumia Egypt metadata
egypt_prods = pd.read_csv("../data/raw/jumia_products.csv", encoding="latin1")

# Extract numeric price and rating
egypt_prods["price_numeric"] = egypt_prods["price"].apply(text_proc.parse_price)
egypt_prods["official_rating"] = (
    egypt_prods["avg_rate"].str.extract(r"(\d+\.\d+|\d+)").astype(float)
)
egypt_prods["reviews_count_numeric"] = (
    egypt_prods["reviews_count"].str.extract(r"(\d+)").fillna(0).astype(int)
)

# 4. Attempt Enrichment via SKU Join
enriched_nlp = nlp_df.merge(
    egypt_prods[["id", "official_rating", "reviews_count_numeric", "price_numeric"]],
    left_on="sku",
    right_on="id",
    how="left",
)

# Diagnostic: Verify Join Success
matches = enriched_nlp["official_rating"].notna().sum()
print(f"✅ NLP Frame Ready: {nlp_df.shape[0]} samples")
print(f"🔗 Successfully merged products: {matches}")
if matches == 0:
    print(
        "⚠️ Note: SKU mismatch detected between Kenya Reviews and Egypt Catalog. Proceeding with Cross-Market Inference."
    )

✅ NLP Frame Ready: 92 samples
🔗 Successfully merged products: 0
⚠️ Note: SKU mismatch detected between Kenya Reviews and Egypt Catalog. Proceeding with Cross-Market Inference.


## 2.2 Vectorization & Training

We use TF-IDF to convert tokens into a numerical matrix, capturing nuanced phrases like "feki sana" (very fake) using bigrams.

In [10]:
# Initialize our model class from src/sentiment_analysis.py
from sklearn.model_selection import train_test_split
sm = SentimentModel(max_features=2500)

# 1. Transform text to features
X_tfidf = sm.prepare_features(nlp_df["tokens"])
y = nlp_df["sentiment_target"]

# 2. Train-Test Split (Stratified to maintain sentiment balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Fit the Classifier
sm.train(X_train, y_train)

# 4. Evaluation (This will print the report)
sm.evaluate(X_test, y_test)

--- Sentiment Engine: Detailed Report ---
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         1
         1.0       0.95      1.00      0.97        18

    accuracy                           0.95        19
   macro avg       0.47      0.50      0.49        19
weighted avg       0.90      0.95      0.92        19



array([[ 0,  1],
       [ 0, 18]])

## 2.3 Cross-Market Inference: Scanning Egypt for Red Flags

**Concept**: Since we cannot join the datasets directly due to SKU differences, we apply the model trained on Kenyan "dissatisfaction patterns" to the Egyptian product catalog. We use Product Names as a proxy for sentiment to identify "Risk Keywords" that contradict high star ratings.

In [13]:
## 2.3 Cross-Market Inference: Scanning Egypt for Red Flags (Sanitized)

# 1. Clean the catalog: Drop rows with missing names to avoid the 'np.nan' crash
egypt_prods_clean = egypt_prods.dropna(subset=["product_name"]).copy()

# 2. Tokenize the product names
egypt_prods_clean["tokens"] = egypt_prods_clean["product_name"].str.lower().str.split()

# 3. Predict Sentiment based on Model Patterns
# Now that we've removed NaNs, transform() will work smoothly
X_egypt = sm.vectorizer.transform(egypt_prods_clean["tokens"])
egypt_prods_clean["predicted_sentiment"] = sm.classifier.predict(X_egypt)

# 4. Detect Asymmetry
asymmetry_egypt = egypt_prods_clean[
    (egypt_prods_clean["official_rating"] >= 4.0)
    & (egypt_prods_clean["predicted_sentiment"] == 0)
].copy()

asymmetry_egypt = asymmetry_egypt.sort_values(
    by="reviews_count_numeric", ascending=False
)

print(
    f"🕵️ Inference Success: Found {len(asymmetry_egypt)} potential cases in the Egypt Market."
)
asymmetry_egypt[
    ["product_name", "official_rating", "reviews_count_numeric", "price_numeric"]
].head(10)

🕵️ Inference Success: Found 0 potential cases in the Egypt Market.


,product_name,official_rating,reviews_count_numeric,price_numeric


## 2.4 Discovery: Identifying Information Asymmetry

### 1. Predict sentiment for the entire enriched dataset


## 2.4.1 Inference: Scanning the Egypt Market for Red Flags

## 1. Prepare the Egypt data for the model
## We use the 'product_name' as the text source since we don't have Egypt reviews yet


In [15]:
egypt_prods['tokens'] = egypt_prods['product_name'].str.lower().str.split()

# 2. Use our trained model to predict sentiment for Egypt
# (We transform the names into the same TF-IDF space as our training data)

X_egypt = sm.vectorizer.transform(egypt_prods['tokens'])
egypt_prods['predicted_sentiment'] = sm.classifier.predict(X_egypt)

# 3. Detect Asymmetry
# High official rating (>= 4.0) but our model flagged the name/description as 'Negative'
asymmetry_egypt = egypt_prods[
    (egypt_prods['official_rating'] >= 4.0) & 
    (egypt_prods['predicted_sentiment'] == 0)
].copy()

# Clean up for display
asymmetry_egypt = asymmetry_egypt.sort_values(by="reviews_count_numeric", ascending=False)

print(f"🕵️ Inference Success: Found {len(asymmetry_egypt)} potential cases in the Egypt Market.")

# 4. View the results
asymmetry_egypt[["product_name", "official_rating", "reviews_count_numeric", "price_numeric"]].head(10)

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [ ]:
# We use the trained vectorizer to transform all tokens
X_all = sm.vectorizer.transform(enriched_nlp["tokens"])
enriched_nlp["predicted_sentiment"] = sm.classifier.predict(X_all)


### 2. Define "Information Asymmetry"
### Scenario: Official rating is High (>= 4.0), but our NLP detects Negative Sentiment (0)

In [ ]:
asymmetry_mask = (enriched_nlp["official_rating"] >= 4.0) & (enriched_nlp["predicted_sentiment"] == 0)
asymmetry_df = enriched_nlp[asymmetry_mask]

print(f"🕵️ Discovery: Found {len(asymmetry_df)} cases of Information Asymmetry.")


🕵️ Discovery: Found 0 cases of Information Asymmetry.



### 3. View the top 'Dangerous' products (high reviews but hidden negativity)


In [ ]:
top_asymmetry = asymmetry_df.sort_values(by="reviews_count_numeric", ascending=False)
top_asymmetry[["product_name", "official_rating", "reviews_count_numeric", "tokens"]].head(10)

,product_name,official_rating,reviews_count_numeric,tokens


In [ ]:
matches = enriched_nlp["official_rating"].notna().sum()
print(f"Number of successfully merged products: {matches}")
print(f"Sample SKUs in reviews: {nlp_df['sku'].head().values}")
print(f"Sample IDs in Egypt CSV: {egypt_prods['id'].head().values}")

Number of successfully merged products: 0
Sample SKUs in reviews: ['VE497DR0RPB0CNAFAMZ' 'VE497DR0RPB0CNAFAMZ' 'VE497DR0RPB0CNAFAMZ'
 'VE497DR0RPB0CNAFAMZ' 'VE497DR0RPB0CNAFAMZ']
Sample IDs in Egypt CSV: ['1' '2' '3' '4' '5']
